# Búsqueda de Publicaciones sobre Alzheimer y Demencia en NCBI

Este notebook utiliza NCBI E-utilities para obtener las últimas publicaciones y papers sobre Alzheimer o demencia desde PubMed.


## Instalación de dependencias

Instalamos las bibliotecas necesarias para trabajar con NCBI E-utilities.


In [22]:
# Instalar dependencias si no están instaladas
!pip install biopython requests xmltodict


## Importación de librerías


In [23]:
from Bio import Entrez
from Bio.Entrez import efetch, esearch, esummary, elink
import xml.etree.ElementTree as ET
from typing import List, Dict, Optional
from datetime import datetime, timedelta
import time
import requests
import io
import re

# Librerías para procesar documentos
try:
    import PyPDF2
    PDF2_AVAILABLE = True
except ImportError:
    PDF2_AVAILABLE = False

try:
    import pdfplumber
    PDFPLUMBER_AVAILABLE = True
except ImportError:
    PDFPLUMBER_AVAILABLE = False

try:
    import fitz  # PyMuPDF
    PYMUPDF_AVAILABLE = True
except ImportError:
    PYMUPDF_AVAILABLE = False

try:
    from docx import Document
    DOCX_AVAILABLE = True
except ImportError:
    DOCX_AVAILABLE = False


## Configuración de NCBI

Configura tu API key de NCBI. Si no tienes una, puedes obtenerla en https://www.ncbi.nlm.nih.gov/account/settings/


In [24]:
NCBI_API_KEY = "#"

Entrez.api_key = NCBI_API_KEY
Entrez.email = "#"


## Método 1: Buscar publicaciones en PubMed

Este método busca publicaciones sobre Alzheimer o demencia en PubMed.


In [25]:
def buscar_publicaciones(
    terminos: List[str] = ["Alzheimer", "dementia"],
    max_resultados: int = 100,
    dias_atras: Optional[int] = None,
    fecha_desde: Optional[str] = None,
    fecha_hasta: Optional[str] = None,
    tipo_publicacion: str = "journal article"
) -> List[str]:
    """
    Busca publicaciones en PubMed sobre los términos especificados.
    
    Parámetros:
    -----------
    terminos : List[str]
        Lista de términos de búsqueda (por defecto ["Alzheimer", "dementia"])
    max_resultados : int
        Número máximo de resultados a retornar (por defecto 100)
    dias_atras : int, opcional
        Número de días hacia atrás desde hoy para filtrar publicaciones
    fecha_desde : str, opcional
        Fecha de inicio en formato YYYY/MM/DD o YYYY/MM o YYYY
    fecha_hasta : str, opcional
        Fecha de fin en formato YYYY/MM/DD o YYYY/MM o YYYY
    tipo_publicacion : str
        Tipo de publicación a buscar (por defecto "journal article")
    
    Retorna:
    --------
    List[str]
        Lista de IDs de PubMed (PMIDs)
    """
    try:
        # Construir query de búsqueda
        query_terms = " OR ".join([f'"{term}"[Title/Abstract]' for term in terminos])
        query = f"({query_terms}) AND {tipo_publicacion}[Publication Type]"
        
        # Agregar filtro de fecha si se especifica
        if dias_atras:
            fecha_hasta = datetime.now().strftime("%Y/%m/%d")
            fecha_desde = (datetime.now() - timedelta(days=dias_atras)).strftime("%Y/%m/%d")
            query += f" AND {fecha_desde}:{fecha_hasta}[Publication Date]"
        elif fecha_desde or fecha_hasta:
            if fecha_desde and fecha_hasta:
                query += f" AND {fecha_desde}:{fecha_hasta}[Publication Date]"
            elif fecha_desde:
                query += f" AND {fecha_desde}:3000/12/31[Publication Date]"
            elif fecha_hasta:
                query += f" AND 1900/01/01:{fecha_hasta}[Publication Date]"
        
        print(f"Buscando publicaciones con query: {query}")
        print(f"Máximo de resultados: {max_resultados}")
        
        # Buscar en PubMed
        handle = esearch(
            db="pubmed",
            term=query,
            retmax=max_resultados,
            retmode="xml",
            sort="pub_date",
            sort_order="desc"
        )
        
        results = Entrez.read(handle)
        handle.close()
        
        pmids = results["IdList"]
        total_encontrados = int(results["Count"])
        
        print(f"Total de publicaciones encontradas: {total_encontrados}")
        print(f"IDs obtenidos: {len(pmids)}")
        
        return pmids
        
    except Exception as e:
        print(f"Error al buscar publicaciones: {e}")
        raise


## Método 2: Obtener detalles de publicaciones

Este método obtiene información detallada de las publicaciones a partir de sus IDs.


In [26]:
def obtener_detalles_publicaciones(
    pmids: List[str],
    batch_size: int = 100
) -> List[Dict]:
    """
    Obtiene detalles completos de publicaciones a partir de sus IDs de PubMed.
    
    Parámetros:
    -----------
    pmids : List[str]
        Lista de IDs de PubMed (PMIDs)
    batch_size : int
        Tamaño del lote para procesar (por defecto 100, máximo recomendado)
    
    Retorna:
    --------
    List[Dict]
        Lista de diccionarios con información de cada publicación
    """
    try:
        todas_publicaciones = []
        
        # Procesar en lotes para evitar límites de la API
        for i in range(0, len(pmids), batch_size):
            batch = pmids[i:i + batch_size]
            print(f"Obteniendo detalles del lote {i//batch_size + 1}/{(len(pmids)-1)//batch_size + 1}...")
            
            # Obtener resúmenes (summaries)
            handle = esummary(db="pubmed", id=",".join(batch), retmode="xml")
            summaries = Entrez.read(handle)
            handle.close()
            
            # Obtener detalles completos (fetch)
            handle = efetch(db="pubmed", id=",".join(batch), retmode="xml")
            records = Entrez.read(handle)
            handle.close()
            
            # Procesar cada publicación
            for j, (summary, record) in enumerate(zip(summaries, records)):
                pub_info = extraer_info_publicacion(summary, record)
                todas_publicaciones.append(pub_info)
            
            # Respetar límite de rate de NCBI (3 requests por segundo sin API key)
            time.sleep(0.34)
        
        print(f"Se obtuvieron detalles de {len(todas_publicaciones)} publicaciones")
        return todas_publicaciones
        
    except Exception as e:
        print(f"Error al obtener detalles: {e}")
        raise


def obtener_valor_seguro(obj, clave, default=""):
    """
    Obtiene un valor de un objeto de Biopython de forma segura.
    Convierte StringElement y otros tipos a tipos nativos de Python.
    """
    try:
        if hasattr(obj, 'get'):
            valor = obj.get(clave, default)
        elif hasattr(obj, '__getitem__'):
            valor = obj[clave] if clave in obj else default
        else:
            return default
        
        # Convertir StringElement a string
        if hasattr(valor, '__str__'):
            return str(valor)
        return valor
    except:
        return default


def extraer_info_publicacion(summary, record) -> Dict:
    """
    Extrae información relevante de una publicación.
    Maneja correctamente los objetos de Biopython (StringElement, ListElement, etc.)
    
    Parámetros:
    -----------
    summary : dict
        Resumen de la publicación desde esummary
    record : dict
        Registro completo de la publicación desde efetch
    
    Retorna:
    --------
    Dict
        Diccionario con información estructurada de la publicación
    """
    try:
        # Información básica - convertir a string
        pmid = str(obtener_valor_seguro(summary, "Id", ""))
        
        # Título
        titulo = ""
        if "Title" in summary:
            titulo = str(summary["Title"])
        elif "MedlineCitation" in record:
            try:
                if "Article" in record["MedlineCitation"]:
                    article = record["MedlineCitation"]["Article"]
                    if "ArticleTitle" in article:
                        titulo = str(article["ArticleTitle"])
            except:
                pass
        
        # Autores
        autores = []
        try:
            # Intentar desde summary primero
            if "AuthorList" in summary:
                author_list = summary["AuthorList"]
                if isinstance(author_list, list):
                    for auth in author_list:
                        try:
                            # Intentar diferentes formas de acceso
                            last = ""
                            first = ""
                            
                            # Forma 1: Name.Last, Name.First
                            if "Name" in auth:
                                name = auth["Name"]
                                if isinstance(name, dict):
                                    last = str(name.get("Last", "")) if hasattr(name, 'get') else str(name.get("Last", ""))
                                    first = str(name.get("First", "")) if hasattr(name, 'get') else str(name.get("First", ""))
                            
                            # Forma 2: LastName, ForeName directamente
                            if not last and "LastName" in auth:
                                last = str(auth["LastName"])
                            if not first and "ForeName" in auth:
                                first = str(auth["ForeName"])
                            
                            if last or first:
                                autor_str = f"{last}, {first}".strip(", ")
                                if autor_str:
                                    autores.append(autor_str)
                        except Exception as e:
                            continue
        except:
            pass
        
        # Si no hay autores en summary, intentar en record
        if not autores and "MedlineCitation" in record:
            try:
                medline = record["MedlineCitation"]
                if "Article" in medline:
                    article = medline["Article"]
                    if "AuthorList" in article:
                        author_list = article["AuthorList"]
                        if isinstance(author_list, list):
                            for auth in author_list:
                                try:
                                    last = str(auth.get("LastName", "")) if hasattr(auth, 'get') else (str(auth["LastName"]) if "LastName" in auth else "")
                                    fore = str(auth.get("ForeName", "")) if hasattr(auth, 'get') else (str(auth["ForeName"]) if "ForeName" in auth else "")
                                    if last or fore:
                                        autor_str = f"{last}, {fore}".strip(", ")
                                        if autor_str:
                                            autores.append(autor_str)
                                except:
                                    continue
            except:
                pass
        
        # Fecha de publicación
        fecha_pub = ""
        try:
            if "PubDate" in summary:
                fecha_pub = str(summary["PubDate"])
            elif "MedlineCitation" in record:
                if "Article" in record["MedlineCitation"]:
                    article = record["MedlineCitation"]["Article"]
                    if "Journal" in article and "JournalIssue" in article["Journal"]:
                        journal_issue = article["Journal"]["JournalIssue"]
                        if "PubDate" in journal_issue:
                            pub_date = journal_issue["PubDate"]
                            if "Year" in pub_date:
                                fecha_pub = str(pub_date["Year"])
                                if "Month" in pub_date:
                                    fecha_pub += f"/{str(pub_date['Month'])}"
                                if "Day" in pub_date:
                                    fecha_pub += f"/{str(pub_date['Day'])}"
        except:
            pass
        
        # Revista
        revista = ""
        try:
            if "Source" in summary:
                revista = str(summary["Source"])
            elif "MedlineCitation" in record:
                if "Article" in record["MedlineCitation"]:
                    article = record["MedlineCitation"]["Article"]
                    if "Journal" in article and "Title" in article["Journal"]:
                        revista = str(article["Journal"]["Title"])
        except:
            pass
        
        # Abstract
        abstract = ""
        try:
            if "Abstract" in summary:
                abstract = str(summary["Abstract"])
            elif "MedlineCitation" in record:
                if "Article" in record["MedlineCitation"]:
                    article = record["MedlineCitation"]["Article"]
                    if "Abstract" in article:
                        abstract_obj = article["Abstract"]
                        if "AbstractText" in abstract_obj:
                            abstract_parts = abstract_obj["AbstractText"]
                            if isinstance(abstract_parts, list):
                                abstract = " ".join([str(part) for part in abstract_parts])
                            else:
                                abstract = str(abstract_parts)
        except:
            pass
        
        # DOI
        doi = ""
        try:
            if "ELocationID" in summary:
                eloc_list = summary["ELocationID"]
                if isinstance(eloc_list, list):
                    for eloc in eloc_list:
                        eloc_str = str(eloc)
                        if eloc_str.startswith("10."):
                            doi = eloc_str
                            break
                else:
                    eloc_str = str(eloc_list)
                    if eloc_str.startswith("10."):
                        doi = eloc_str
        except:
            pass
        
        # URL de PubMed
        url_pubmed = f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else ""
        
        # Palabras clave (MeSH terms)
        mesh_terms = []
        try:
            if "MedlineCitation" in record:
                medline = record["MedlineCitation"]
                if "MeshHeadingList" in medline:
                    mesh_list = medline["MeshHeadingList"]
                    if isinstance(mesh_list, list):
                        for mesh in mesh_list:
                            try:
                                if "DescriptorName" in mesh:
                                    mesh_term = str(mesh["DescriptorName"])
                                    mesh_terms.append(mesh_term)
                            except:
                                continue
        except:
            pass
        
        return {
            "pmid": pmid,
            "titulo": titulo,
            "autores": autores,
            "fecha_publicacion": fecha_pub,
            "revista": revista,
            "abstract": abstract,
            "doi": doi,
            "url_pubmed": url_pubmed,
            "mesh_terms": mesh_terms
        }
        
    except Exception as e:
        print(f"Error al extraer información: {e}")
        import traceback
        traceback.print_exc()
        return {
            "pmid": "",
            "titulo": "",
            "autores": [],
            "fecha_publicacion": "",
            "revista": "",
            "abstract": "",
            "doi": "",
            "url_pubmed": "",
            "mesh_terms": []
        }


## Método 4: Obtener URLs de documentos completos

Este método obtiene las URLs de los documentos completos (PDF, Word) disponibles para una publicación.


In [27]:
def obtener_urls_documento(pmid: str) -> Dict[str, str]:
    """
    Obtiene las URLs de documentos completos disponibles para una publicación.
    
    Parámetros:
    -----------
    pmid : str
        ID de PubMed de la publicación
    
    Retorna:
    --------
    Dict[str, str]
        Diccionario con URLs disponibles:
        - pmc_pdf: URL del PDF en PMC (si está disponible)
        - pmc_html: URL HTML en PMC
        - doi_url: URL del DOI
        - publisher_url: URL del editor
    """
    urls = {
        "pmc_pdf": "",
        "pmc_html": "",
        "doi_url": "",
        "publisher_url": ""
    }
    
    try:
        # Obtener información de PMC (PubMed Central)
        handle = elink(dbfrom="pubmed", db="pmc", id=pmid)
        record = Entrez.read(handle)
        handle.close()
        
        if record and len(record) > 0:
            if "LinkSetDb" in record[0] and len(record[0]["LinkSetDb"]) > 0:
                pmc_ids = record[0]["LinkSetDb"][0].get("Link", [])
                if pmc_ids:
                    pmc_id = str(pmc_ids[0].get("Id", ""))
                    if pmc_id:
                        urls["pmc_pdf"] = f"https://www.ncbi.nlm.nih.gov/pmc/articles/PMC{pmc_id}/pdf/"
                        urls["pmc_html"] = f"https://www.ncbi.nlm.nih.gov/pmc/articles/PMC{pmc_id}/"
        
        # Obtener DOI y construir URL
        handle = efetch(db="pubmed", id=pmid, retmode="xml")
        record = Entrez.read(handle)
        handle.close()
        
        if record and "PubmedData" in record[0]:
            pubmed_data = record[0]["PubmedData"]
            if "ArticleIdList" in pubmed_data:
                for article_id in pubmed_data["ArticleIdList"]:
                    if str(article_id).startswith("10."):
                        doi = str(article_id)
                        urls["doi_url"] = f"https://doi.org/{doi}"
                    elif hasattr(article_id, 'attributes') and article_id.attributes.get("IdType") == "doi":
                        doi = str(article_id)
                        urls["doi_url"] = f"https://doi.org/{doi}"
        
    except Exception as e:
        print(f"Error al obtener URLs del documento: {e}")
    
    return urls


## Método 5: Convertir PDF a texto

Este método descarga un PDF y lo convierte a texto plano.


In [28]:
def pdf_a_texto(url_pdf: str, timeout: int = 30) -> str:
    """
    Descarga un PDF desde una URL y lo convierte a texto plano.
    
    Parámetros:
    -----------
    url_pdf : str
        URL del PDF a descargar
    timeout : int
        Tiempo máximo de espera en segundos (por defecto 30)
    
    Retorna:
    --------
    str
        Texto extraído del PDF
    """
    try:
        # Descargar el PDF
        print(f"Descargando PDF desde: {url_pdf}")
        response = requests.get(url_pdf, timeout=timeout, stream=True)
        response.raise_for_status()
        
        pdf_bytes = response.content
        
        texto_completo = ""
        
        # Intentar con PyMuPDF (más rápido y mejor calidad)
        if PYMUPDF_AVAILABLE:
            try:
                doc = fitz.open(stream=pdf_bytes, filetype="pdf")
                for page_num in range(len(doc)):
                    page = doc[page_num]
                    texto_completo += page.get_text()
                doc.close()
                print(f"PDF convertido a texto usando PyMuPDF ({len(doc)} páginas)")
                return texto_completo
            except Exception as e:
                print(f"Error con PyMuPDF: {e}, intentando con otra librería...")
        
        # Intentar con pdfplumber (mejor para tablas)
        if PDFPLUMBER_AVAILABLE:
            try:
                with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
                    for page in pdf.pages:
                        texto = page.extract_text()
                        if texto:
                            texto_completo += texto + "\n"
                print(f"PDF convertido a texto usando pdfplumber")
                return texto_completo
            except Exception as e:
                print(f"Error con pdfplumber: {e}, intentando con PyPDF2...")
        
        # Intentar con PyPDF2 (fallback)
        if PDF2_AVAILABLE:
            try:
                pdf_file = io.BytesIO(pdf_bytes)
                pdf_reader = PyPDF2.PdfReader(pdf_file)
                for page_num in range(len(pdf_reader.pages)):
                    page = pdf_reader.pages[page_num]
                    texto_completo += page.extract_text() + "\n"
                print(f"PDF convertido a texto usando PyPDF2 ({len(pdf_reader.pages)} páginas)")
                return texto_completo
            except Exception as e:
                print(f"Error con PyPDF2: {e}")
        
        if not texto_completo:
            raise Exception("No se pudo extraer texto del PDF. Asegúrate de tener instalada al menos una de: pymupdf, pdfplumber, PyPDF2")
        
        return texto_completo
        
    except requests.exceptions.RequestException as e:
        print(f"Error al descargar el PDF: {e}")
        raise
    except Exception as e:
        print(f"Error al convertir PDF a texto: {e}")
        raise


## Método 6: Convertir Word a texto

Este método descarga un documento Word y lo convierte a texto plano.


In [29]:
def word_a_texto(url_docx: str, timeout: int = 30) -> str:
    """
    Descarga un documento Word desde una URL y lo convierte a texto plano.
    
    Parámetros:
    -----------
    url_docx : str
        URL del documento Word a descargar
    timeout : int
        Tiempo máximo de espera en segundos (por defecto 30)
    
    Retorna:
    --------
    str
        Texto extraído del documento Word
    """
    if not DOCX_AVAILABLE:
        raise ImportError("python-docx no está instalado. Instálalo con: pip install python-docx")
    
    try:
        # Descargar el documento
        print(f"Descargando documento Word desde: {url_docx}")
        response = requests.get(url_docx, timeout=timeout, stream=True)
        response.raise_for_status()
        
        docx_bytes = response.content
        
        # Abrir el documento
        doc = Document(io.BytesIO(docx_bytes))
        
        # Extraer texto de todos los párrafos
        texto_completo = []
        for paragraph in doc.paragraphs:
            texto_completo.append(paragraph.text)
        
        # Extraer texto de las tablas
        for table in doc.tables:
            for row in table.rows:
                for cell in row.cells:
                    texto_completo.append(cell.text)
        
        texto = "\n".join(texto_completo)
        print(f"Documento Word convertido a texto ({len(doc.paragraphs)} párrafos)")
        return texto
        
    except requests.exceptions.RequestException as e:
        print(f"Error al descargar el documento Word: {e}")
        raise
    except Exception as e:
        print(f"Error al convertir Word a texto: {e}")
        raise


## Método 7: Obtener texto completo de una publicación

Este método obtiene el abstract y el documento completo (si está disponible) de una publicación.


In [30]:
def obtener_texto_completo_publicacion(
    pub_info: Dict,
    preferir_pmc: bool = True
) -> Dict[str, str]:
    """
    Obtiene el texto completo de una publicación, incluyendo abstract y documento completo.
    
    Parámetros:
    -----------
    pub_info : Dict
        Diccionario con información de la publicación (debe incluir 'pmid')
    preferir_pmc : bool
        Si es True, intenta obtener el PDF de PMC primero (por defecto True)
    
    Retorna:
    --------
    Dict[str, str]
        Diccionario con:
        - abstract: Resumen de la publicación
        - texto_completo: Texto completo del documento (si está disponible)
        - fuente: Fuente del texto completo ("pmc_pdf", "pmc_html", "doi", "none")
        - url_usada: URL del documento usado
    """
    resultado = {
        "abstract": pub_info.get("abstract", ""),
        "texto_completo": "",
        "fuente": "none",
        "url_usada": ""
    }
    
    try:
        pmid = pub_info.get("pmid", "")
        if not pmid:
            print("No se proporcionó PMID")
            return resultado
        
        # Obtener URLs disponibles
        urls = obtener_urls_documento(pmid)
        
        # Intentar obtener el documento completo
        if preferir_pmc and urls["pmc_pdf"]:
            try:
                print(f"Intentando obtener PDF desde PMC: {urls['pmc_pdf']}")
                texto = pdf_a_texto(urls["pmc_pdf"])
                resultado["texto_completo"] = texto
                resultado["fuente"] = "pmc_pdf"
                resultado["url_usada"] = urls["pmc_pdf"]
                print("✓ Texto completo obtenido desde PMC PDF")
                return resultado
            except Exception as e:
                print(f"No se pudo obtener PDF de PMC: {e}")
        
        # Intentar con DOI (puede requerir acceso al editor)
        if urls["doi_url"]:
            try:
                print(f"Intentando obtener documento desde DOI: {urls['doi_url']}")
                # Nota: Muchos DOI requieren acceso institucional o pago
                # Esto es solo un intento básico
                response = requests.get(urls["doi_url"], timeout=10, allow_redirects=True)
                final_url = response.url
                
                # Si la URL final es un PDF, intentar descargarlo
                if final_url.endswith(".pdf") or "pdf" in final_url.lower():
                    texto = pdf_a_texto(final_url)
                    resultado["texto_completo"] = texto
                    resultado["fuente"] = "doi_pdf"
                    resultado["url_usada"] = final_url
                    print("✓ Texto completo obtenido desde DOI PDF")
                    return resultado
            except Exception as e:
                print(f"No se pudo obtener documento desde DOI: {e}")
        
        # Si no se pudo obtener el documento completo, al menos retornar el abstract
        if resultado["abstract"]:
            print("⚠ Solo se pudo obtener el abstract, no el documento completo")
        else:
            print("⚠ No se pudo obtener ni el abstract ni el documento completo")
        
    except Exception as e:
        print(f"Error al obtener texto completo: {e}")
    
    return resultado


### Ejemplo 5: Obtener texto completo de una publicación


In [31]:
publicaciones_ejemplo = obtener_ultimas_publicaciones(
    terminos=["Alzheimer"],
    max_resultados=3,
    dias_atras=60
)

if publicaciones_ejemplo:
    primera_pub = publicaciones_ejemplo[0]
    print(f"PMID: {primera_pub['pmid']}")
    print(f"Título: {primera_pub['titulo']}\n")
    
    texto_completo = obtener_texto_completo_publicacion(primera_pub)
    
    print(f"\n=== RESUMEN ===")
    print(texto_completo['abstract'][:500] if texto_completo['abstract'] else "No disponible")
    
    if texto_completo['texto_completo']:
        print(f"\n=== TEXTO COMPLETO ===")
        print(f"Fuente: {texto_completo['fuente']}")
        print(f"URL: {texto_completo['url_usada']}")
        print(f"Longitud del texto: {len(texto_completo['texto_completo'])} caracteres")
        print(f"\nPrimeros 1000 caracteres:\n{texto_completo['texto_completo'][:1000]}...")
    else:
        print("\n⚠ No se pudo obtener el documento completo")


Buscando las últimas publicaciones sobre: Alzheimer
Período: últimos 60 días
------------------------------------------------------------
Buscando publicaciones con query: ("Alzheimer"[Title/Abstract]) AND journal article[Publication Type] AND 2025/10/08:2025/12/07[Publication Date]
Máximo de resultados: 3
Total de publicaciones encontradas: 3449
IDs obtenidos: 3
Obteniendo detalles del lote 1/1...
Se obtuvieron detalles de 2 publicaciones
PMID: 41306192
Título: Control analysis of deep brain stimulation and optogenetics for Alzheimer's disease under the computational cortex model.

Error al obtener URLs del documento: NCBI C++ Exception:
    Error: TXCLIENT(CException::eUnknown) "/pubmed_gen/rbuild/version/20250923/entrez/2.20/src/internal/txclient/TxClient.cpp", line 1100: ncbi::CTxRawClientImpl::readAll() --- Read failed: EOF (the other side has unexpectedly closed connection), peer: 130.14.22.35:8064

⚠ No se pudo obtener ni el abstract ni el documento completo

=== RESUMEN ===
No 

### Ejemplo 6: Obtener URLs de documentos disponibles


In [32]:
pmid_ejemplo = "41306192"

urls = obtener_urls_documento(pmid_ejemplo)

print(f"URLs disponibles para PMID {pmid_ejemplo}:")
print(f"PMC PDF: {urls['pmc_pdf'] if urls['pmc_pdf'] else 'No disponible'}")
print(f"PMC HTML: {urls['pmc_html'] if urls['pmc_html'] else 'No disponible'}")
print(f"DOI URL: {urls['doi_url'] if urls['doi_url'] else 'No disponible'}")

if urls['pmc_pdf']:
    print(f"\nDescargando y convirtiendo PDF...")
    try:
        texto_pdf = pdf_a_texto(urls['pmc_pdf'])
        print(f"✓ PDF convertido exitosamente")
        print(f"Longitud del texto: {len(texto_pdf)} caracteres")
        print(f"\nPrimeros 500 caracteres:\n{texto_pdf[:500]}...")
    except Exception as e:
        print(f"Error: {e}")


Error al obtener URLs del documento: 0
URLs disponibles para PMID 41306192:
PMC PDF: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC12644327/pdf/
PMC HTML: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC12644327/
DOI URL: No disponible

Descargando y convirtiendo PDF...
Descargando PDF desde: https://www.ncbi.nlm.nih.gov/pmc/articles/PMC12644327/pdf/
Error al descargar el PDF: 403 Client Error: Forbidden for url: https://pmc.ncbi.nlm.nih.gov/articles/PMC12644327/pdf/
Error: 403 Client Error: Forbidden for url: https://pmc.ncbi.nlm.nih.gov/articles/PMC12644327/pdf/


In [33]:
def obtener_ultimas_publicaciones(
    terminos: List[str] = ["Alzheimer", "dementia"],
    max_resultados: int = 50,
    dias_atras: int = 30,
    tipo_publicacion: str = "journal article"
) -> List[Dict]:
    """
    Obtiene las últimas publicaciones sobre los términos especificados.
    
    Parámetros:
    -----------
    terminos : List[str]
        Lista de términos de búsqueda (por defecto ["Alzheimer", "dementia"])
    max_resultados : int
        Número máximo de resultados a retornar (por defecto 50)
    dias_atras : int
        Número de días hacia atrás desde hoy (por defecto 30)
    tipo_publicacion : str
        Tipo de publicación a buscar (por defecto "journal article")
    
    Retorna:
    --------
    List[Dict]
        Lista de diccionarios con información completa de cada publicación
    """
    print(f"Buscando las últimas publicaciones sobre: {', '.join(terminos)}")
    print(f"Período: últimos {dias_atras} días")
    print("-" * 60)
    
    # Buscar IDs
    pmids = buscar_publicaciones(
        terminos=terminos,
        max_resultados=max_resultados,
        dias_atras=dias_atras,
        tipo_publicacion=tipo_publicacion
    )
    
    if not pmids:
        print("No se encontraron publicaciones")
        return []
    
    # Obtener detalles
    publicaciones = obtener_detalles_publicaciones(pmids)
    
    return publicaciones


## Ejemplos de uso

A continuación se muestran ejemplos de cómo usar los métodos para obtener publicaciones.


### Ejemplo 1: Obtener últimas publicaciones sobre Alzheimer


In [34]:
publicaciones = obtener_ultimas_publicaciones(
    terminos=["Alzheimer"],
    max_resultados=10,
    dias_atras=30
)

print(f"\nSe encontraron {len(publicaciones)} publicaciones\n")
print(publicaciones)
for i, pub in enumerate(publicaciones[:5], 1):
    print(f"{i}. {pub['titulo']}")
    print(f"   Autores: {', '.join(pub['autores'][:3])}..." if len(pub['autores']) > 3 else f"   Autores: {', '.join(pub['autores'])}")
    print(f"   Fecha: {pub['fecha_publicacion']}")
    print(f"   Revista: {pub['revista']}")
    print(f"   PMID: {pub['pmid']}")
    print(f"   URL: {pub['url_pubmed']}")
    print()


Buscando las últimas publicaciones sobre: Alzheimer
Período: últimos 30 días
------------------------------------------------------------
Buscando publicaciones con query: ("Alzheimer"[Title/Abstract]) AND journal article[Publication Type] AND 2025/11/07:2025/12/07[Publication Date]
Máximo de resultados: 10
Total de publicaciones encontradas: 1627
IDs obtenidos: 10
Obteniendo detalles del lote 1/1...
Se obtuvieron detalles de 2 publicaciones

Se encontraron 2 publicaciones

[{'pmid': '41306192', 'titulo': "Control analysis of deep brain stimulation and optogenetics for Alzheimer's disease under the computational cortex model.", 'autores': [], 'fecha_publicacion': '2026 Dec', 'revista': 'Cogn Neurodyn', 'abstract': '', 'doi': '', 'url_pubmed': 'https://pubmed.ncbi.nlm.nih.gov/41306192/', 'mesh_terms': []}, {'pmid': '41221324', 'titulo': 'Research on the classification of EEG signals for dementia and its interpretability using the GWOCS agorithm.', 'autores': [], 'fecha_publicacion': '20

### Ejemplo 2: Buscar publicaciones sobre demencia con filtros específicos


In [40]:
pmids = buscar_publicaciones(
    terminos=["dementia", "cognitive impairment"],
    max_resultados=20,
    dias_atras=365
)

publicaciones_demencia = obtener_detalles_publicaciones(pmids)

print(f"\nPublicaciones sobre demencia (últimos 365 días): {len(publicaciones_demencia)}\n")
for i, pub in enumerate(publicaciones_demencia[:5], 1):
    print(f"{i}. {pub['titulo']}")
    if pub['abstract']:
        abstract_corto = pub['abstract'][:200] + "..." if len(pub['abstract']) > 200 else pub['abstract']
        print(f"   Abstract: {abstract_corto}")
    print()


Buscando publicaciones con query: ("dementia"[Title/Abstract] OR "cognitive impairment"[Title/Abstract]) AND journal article[Publication Type] AND 2024/12/07:2025/12/07[Publication Date]
Máximo de resultados: 20
Total de publicaciones encontradas: 21232
IDs obtenidos: 20
Obteniendo detalles del lote 1/1...
Se obtuvieron detalles de 2 publicaciones

Publicaciones sobre demencia (últimos 365 días): 2

1. Control analysis of deep brain stimulation and optogenetics for Alzheimer's disease under the computational cortex model.

2. Research on the classification of EEG signals for dementia and its interpretability using the GWOCS agorithm.



### Ejemplo 3: Buscar publicaciones por rango de fechas


In [36]:
pmids_fechas = buscar_publicaciones(
    terminos=["Alzheimer disease"],
    max_resultados=15,
    fecha_desde="2024/01/01",
    fecha_hasta="2024/12/31"
)

publicaciones_2024 = obtener_detalles_publicaciones(pmids_fechas)

print(f"\nPublicaciones de 2024: {len(publicaciones_2024)}\n")
for pub in publicaciones_2024[:3]:
    print(f"Título: {pub['titulo']}")
    print(f"Fecha: {pub['fecha_publicacion']}")
    print(f"DOI: {pub['doi'] if pub['doi'] else 'No disponible'}")
    print(f"MeSH Terms: {', '.join(pub['mesh_terms'][:5])}")
    print()


Buscando publicaciones con query: ("Alzheimer disease"[Title/Abstract]) AND journal article[Publication Type] AND 2024/01/01:2024/12/31[Publication Date]
Máximo de resultados: 15
Total de publicaciones encontradas: 1051
IDs obtenidos: 15
Obteniendo detalles del lote 1/1...
Se obtuvieron detalles de 2 publicaciones

Publicaciones de 2024: 2

Título: The Comparison of Oropharyngeal Dysphagia in Alzheimer's Disease versus Older Adults with Presbyphagia.
Fecha: 2025 Aug
DOI: No disponible
MeSH Terms: 

Título: Distortion errors characterise visuo-constructive performance in Huntington's disease.
Fecha: 2025 Aug
DOI: No disponible
MeSH Terms: 



### Ejemplo 4: Exportar resultados a formato estructurado


In [37]:
import json

publicaciones_export = obtener_ultimas_publicaciones(
    terminos=["Alzheimer", "dementia"],
    max_resultados=5,
    dias_atras=7
)

with open("publicaciones_alzheimer.json", "w", encoding="utf-8") as f:
    json.dump(publicaciones_export, f, indent=2, ensure_ascii=False)

print(f"Se exportaron {len(publicaciones_export)} publicaciones a publicaciones_alzheimer.json")

for pub in publicaciones_export:
    print(f"\nPMID: {pub['pmid']}")
    print(f"Título: {pub['titulo']}")
    print(f"Autores: {', '.join(pub['autores'][:5])}")
    print(f"Revista: {pub['revista']}")
    print(f"URL: {pub['url_pubmed']}")


Buscando las últimas publicaciones sobre: Alzheimer, dementia
Período: últimos 7 días
------------------------------------------------------------
Buscando publicaciones con query: ("Alzheimer"[Title/Abstract] OR "dementia"[Title/Abstract]) AND journal article[Publication Type] AND 2025/11/30:2025/12/07[Publication Date]
Máximo de resultados: 5
Total de publicaciones encontradas: 919
IDs obtenidos: 5
Obteniendo detalles del lote 1/1...
Se obtuvieron detalles de 2 publicaciones
Se exportaron 2 publicaciones a publicaciones_alzheimer.json

PMID: 41329157
Título: Amyloidosis of bridging veins is a pathologic feature of Alzheimer's disease.
Autores: 
Revista: J Exp Med
URL: https://pubmed.ncbi.nlm.nih.gov/41329157/

PMID: 41348999
Título: Impact of Cerebral Microbleeds on Tau-Associated Cognitive and Structural Decline.
Autores: 
Revista: Neurology
URL: https://pubmed.ncbi.nlm.nih.gov/41348999/
